In [7]:
import os

API_KEY = os.getenv("OPENAQ_API_KEY")
BASE_URL = "https://api.openaq.org/v3"

headers = {
    "X-API-Key": API_KEY
}

In [8]:
import requests
import pandas as pd

def get_locations_nepal(limit=1000):
    url = f"{BASE_URL}/locations"
    params = {
        "iso": "NP",
        "limit": limit,
        "page": 1
    }

    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()
    data = r.json()

    return data["results"]

locations = get_locations_nepal()
len(locations)

76

In [15]:
rows = []

for loc in locations:
    rows.append({
        "location_id": loc["id"],
        "name": loc["name"],
        "locality": loc.get("locality"),
        "country": loc["country"]["code"],
        "latitude": loc["coordinates"]["latitude"] if loc.get("coordinates") else None,
        "longitude": loc["coordinates"]["longitude"] if loc.get("coordinates") else None,
        "timezone": loc.get("timezone"),
        "datetime_first": loc["datetimeFirst"]["utc"] if loc.get("datetimeFirst") else None,
        "datetime_last": loc["datetimeLast"]["utc"] if loc.get("datetimeLast") else None,
        "is_monitor": loc.get("isMonitor"),
        "provider": loc["provider"]["name"] if loc.get("provider") else None,
        "owner": loc["owner"]["name"] if loc.get("owner") else None,
        "sensors": loc.get("sensors")
    })

locations_df = pd.DataFrame(rows)
locations_df["name"].to_list()

['Embassy Kathmandu',
 'Phora Durbar Kathman',
 'Dhathutole, Handigaun',
 'Dabali, Handigaun',
 'Gaushala Chowk (SC-01) - GD Labs',
 'Baluwatar (SC-02) - GD Labs',
 'Lagankhel (SC - 05) - GD Labs',
 'Sifal(SC-03)- GD Labs',
 'Lamtangil (SC-04)- GD Labs',
 'Pulchowk (SC-44) - GD Labs',
 'Nakhipot (SC-08) - GD Labs',
 'Sunakothi (SC - 06) - GD Labs',
 'Chovar (SC - 07) - GD Labs',
 'Taudaha (SC - 09) - GD Labs',
 'CEN-SR-25: Patako Chowk, Patan Durbar Square',
 'Bharatpur Ward no 27 office Meghauli',
 'Farsidol Relocated',
 'Hetauda Sub-Metropolitan City Office (CEN-SR-15)',
 'CEN-SR-02: Farsidol Brick Factories',
 'Hetauda Udhyog Sang Office (CEN-SR-18)',
 'Shantichowk (SC-11) - GD Labs',
 'Gothatar (SC-12) - GD Labs',
 'Gokarneshwor (SC-13) - GD Labs',
 'CEN_SR-09: Dhangadhdi Sub-metropolitan City Office',
 'CEN-SR-14: Dhangadhi Sub-metropolitan Ward 8',
 'Mahankal (SC-16) - GD Labs',
 'Chhetrapati (SC - 19) - GD Labs',
 'Tarakeswor (SC-14)-GD Labs',
 'Golfutar (SC- 17) - GD Labs',
 'T

In [11]:
locations_df.to_csv("openaq_nepal_locations.csv", index=False)

In [12]:
sensor_rows = []

for loc in locations:
    for sensor in loc.get("sensors", []):
        parameter = sensor.get("parameter", {})
        sensor_rows.append({
            "location_id": loc["id"],
            "location_name": loc["name"],
            "sensor_id": sensor["id"],
            "sensor_name": sensor["name"],
            "parameter": parameter.get("name"),
            "display_name": parameter.get("displayName"),
            "units": parameter.get("units"),
            "latitude": loc["coordinates"]["latitude"] if loc.get("coordinates") else None,
            "longitude": loc["coordinates"]["longitude"] if loc.get("coordinates") else None,
            "datetime_first": loc["datetimeFirst"]["utc"] if loc.get("datetimeFirst") else None,
            "datetime_last": loc["datetimeLast"]["utc"] if loc.get("datetimeLast") else None,
        })

sensors_df = pd.DataFrame(sensor_rows)
sensors_df

,location_id,location_name,sensor_id,sensor_name,parameter,display_name,units,latitude,longitude,datetime_first,datetime_last
0,3459,Embassy Kathmandu,7713,o3 ppm,o3,O₃,ppm,27.738703,85.336206,2017-03-03T00:00:00Z,2026-05-12T06:15:00Z
1,3459,Embassy Kathmandu,7710,pm25 µg/m³,pm25,PM2.5,µg/m³,27.738703,85.336206,2017-03-03T00:00:00Z,2026-05-12T06:15:00Z
2,3460,Phora Durbar Kathman,7712,o3 ppm,o3,O₃,ppm,27.712464,85.315703,2017-03-03T00:00:00Z,2025-03-24T14:15:00Z
3,3460,Phora Durbar Kathman,7711,pm25 µg/m³,pm25,PM2.5,µg/m³,27.712464,85.315703,2017-03-03T00:00:00Z,2025-03-24T14:15:00Z
4,1236017,"Dhathutole, Handigaun",8462455,pm1 µg/m³,pm1,PM1,µg/m³,27.727502,85.330135,2023-07-15T13:00:00Z,2026-05-19T05:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...
371,6336000,Herne Katha,16162467,pm1 µg/m³,pm1,PM1,µg/m³,27.683824,85.310177,2026-05-05T13:00:00Z,2026-05-19T05:00:00Z
372,6336000,Herne Katha,16162468,pm25 µg/m³,pm25,PM2.5,µg/m³,27.683824,85.310177,2026-05-05T13:00:00Z,2026-05-19T05:00:00Z
373,6336000,Herne Katha,16162469,relativehumidity %,relativehumidity,RH,%,27.683824,85.310177,2026-05-05T13:00:00Z,2026-05-19T05:00:00Z
374,6336000,Herne Katha,16162470,temperature c,temperature,Temperature (C),c,27.683824,85.310177,2026-05-05T13:00:00Z,2026-05-19T05:00:00Z


In [16]:
import pandas as pd

LOCATIONS_CSV = "openaq_nepal_locations.csv"

df = pd.read_csv(LOCATIONS_CSV)

# Basic cleanup
df["name_clean"] = df["name"].astype(str).str.lower()
df["locality_clean"] = df["locality"].astype(str).str.lower()

# Approx Kathmandu Valley bounding box
# Adjust if needed
KTM_BBOX = {
    "min_lat": 27.50,
    "max_lat": 27.90,
    "min_lon": 85.10,
    "max_lon": 85.60,
}

df["in_kathmandu_valley_bbox"] = (
    df["latitude"].between(KTM_BBOX["min_lat"], KTM_BBOX["max_lat"])
    & df["longitude"].between(KTM_BBOX["min_lon"], KTM_BBOX["max_lon"])
)

# Rough Terai latitude band + known Terai city keywords
terai_keywords = [
    "biratnagar", "janakpur", "birgunj", "bhairahawa", "butwal",
    "nepalgunj", "dhangadhi", "mahendranagar", "bharatpur",
    "hetauda", "simara", "lumbini", "kapilvastu", "siddharthanagar",
    "rajbiraj", "siraha", "saptari", "rupandehi", "parsa", "bara",
    "morang", "sunsari", "kailali", "kanchanpur", "banke", "bardiya",
    "chitwan", "nawalparasi"
]

df["matches_terai_keyword"] = df["name_clean"].str.contains("|".join(terai_keywords), na=False) | \
                              df["locality_clean"].str.contains("|".join(terai_keywords), na=False)

# Very rough geographic Terai filter
df["likely_terai_latlon"] = (
    df["latitude"].between(26.2, 29.0)
    & df["longitude"].between(80.0, 88.5)
    & ~df["in_kathmandu_valley_bbox"]
)

df["likely_terai"] = df["matches_terai_keyword"] | df["likely_terai_latlon"]

# Out of valley only
out_valley = df[~df["in_kathmandu_valley_bbox"]].copy()

# Sort Terai-looking locations first
out_valley = out_valley.sort_values(
    by=["likely_terai", "matches_terai_keyword", "latitude"],
    ascending=[False, False, True]
)

cols = [
    "location_id", "name", "locality", "latitude", "longitude",
    "datetime_first", "datetime_last", "provider", "owner",
    "likely_terai", "matches_terai_keyword"
]

print("\n=== OUT-OF-VALLEY LOCATIONS, TERAI CANDIDATES FIRST ===\n")

for _, row in out_valley[cols].iterrows():
    print(
        f'{row["location_id"]:<8} | '
        f'{row["name"]} | '
        f'locality={row["locality"]} | '
        f'lat={row["latitude"]}, lon={row["longitude"]} | '
        f'terai={row["likely_terai"]}'
    )

# Optional: save for manual review
out_valley[cols].to_csv("openaq_nepal_out_valley_review.csv", index=False)

print("\nSaved: openaq_nepal_out_valley_review.csv")


=== OUT-OF-VALLEY LOCATIONS, TERAI CANDIDATES FIRST ===

6143672  | CEN-SR-06: Biratnagar Metropolitan City ward no 9 Office | locality=nan | lat=26.45611111, lon=87.28388889 | terai=True
6141726  | CEN-SR-03:Biratnagar metropolitan city office  | locality=nan | lat=26.465215, lon=87.283333 | terai=True
6138104  | CEN-SR-11: Janakpurdham Sub-metropolitian City Office | locality=nan | lat=26.7303, lon=85.9317 | terai=True
6151924  | CEN-SR-20: Janakpurdham SMC-08, Rajarshi Janak University | locality=nan | lat=26.739072, lon=85.916906 | terai=True
6135972  | CEN-SR-13: Birgunj Metropolitan City Office | locality=nan | lat=27.0118695, lon=84.8709016 | terai=True
6122103  | CEN-SR-04: Birgunj Metropolitan City Ward No 25 | locality=nan | lat=27.0264126, lon=84.852162 | terai=True
5710078  | Hetauda Udhyog Sang Office (CEN-SR-18) | locality=nan | lat=27.40233, lon=85.02521 | terai=True
5709621  | Hetauda Sub-Metropolitan City Office (CEN-SR-15) | locality=nan | lat=27.43358, lon=85.03828 